# 02 - Training YOLOv8n (Baseline 1)

Notebook ini melatih YOLOv8n standar tanpa modifikasi sebagai baseline pertama.
Urutan: setup -> load config -> training -> simpan hasil.

## Setup Environment

In [4]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    PROJECT_ROOT = '/content/drive/MyDrive/malaria-detection'
    os.chdir(PROJECT_ROOT)
else:
    import os
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

print('working directory:', os.getcwd())

working directory: d:\malaria-detection


In [5]:
!pip install -q ultralytics


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
import json
from pathlib import Path

import torch
from ultralytics import YOLO

In [7]:
# cek gpu tersedia atau tidak
print('torch version :', torch.__version__)
print('cuda available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('gpu           :', torch.cuda.get_device_name(0))
    DEVICE = 0
else:
    print('gpu tidak ditemukan, training pakai cpu (lebih lambat)')
    DEVICE = 'cpu'

torch version : 2.5.1+cu121
cuda available: True
gpu           : NVIDIA GeForce RTX 3060 Laptop GPU


In [8]:
# load summary dari notebook 01 untuk verifikasi dataset sudah siap
summary_path = os.path.join(PROJECT_ROOT, 'notebook01_summary.json')

if not os.path.exists(summary_path):
    raise FileNotFoundError('notebook01_summary.json tidak ditemukan, pastikan notebook 01 sudah dijalankan sampai selesai')

with open(summary_path) as f:
    summary = json.load(f)

print('status notebook 01:', summary['status'])
print('train            :', summary['total_train'], 'gambar')
print('val              :', summary['total_val'], 'gambar')
print('test             :', summary['total_test'], 'gambar')
print('yaml path        :', summary['yaml_path'])

YAML_PATH = summary['yaml_path']

status notebook 01: done
train            : 21676 gambar
val              : 5606 gambar
test             : 2803 gambar
yaml path        : d:\malaria-detection\malaria.yaml


## Training YOLOv8n

In [10]:
# folder untuk menyimpan hasil training
SAVE_DIR = os.path.join(PROJECT_ROOT, 'runs', 'yolov8n')
os.makedirs(SAVE_DIR, exist_ok=True)

print('hasil training akan disimpan di:', SAVE_DIR)

hasil training akan disimpan di: d:\malaria-detection\runs\yolov8n


In [7]:
# load model yolov8n pretrained dari imagenet
model = YOLO('yolov8n.pt')

print('arsitektur model:')
print('  nama    :', model.info())

arsitektur model:
YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs
  nama    : (129, 3157200, 0, 8.8575488)


In [8]:
# training
# semua hyperparameter di sini harus sama persis dengan notebook 03 dan 04
# supaya perbandingan antar model fair
results = model.train(
    data      = YAML_PATH,
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,
    optimizer = 'AdamW',
    lr0       = 0.001,
    patience  = 20,        # early stopping jika val loss tidak turun 20 epoch
    device    = DEVICE,
    project   = SAVE_DIR,
    name      = 'train',
    save      = True,
    save_period = 10,      # simpan checkpoint tiap 10 epoch
    exist_ok  = True,
    verbose   = True,
)

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.56  Python-3.12.3 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\malaria-detection\malaria.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, 

## Evaluasi pada Test Set

In [11]:
# load best weights hasil training
best_weights = os.path.join(SAVE_DIR, 'train', 'weights', 'best.pt')

if not os.path.exists(best_weights):
    raise FileNotFoundError('best.pt tidak ditemukan, pastikan training sudah selesai')

model_best = YOLO(best_weights)
print('best weights loaded dari:', best_weights)

best weights loaded dari: d:\malaria-detection\runs\yolov8n\train\weights\best.pt


In [12]:
# evaluasi di test set
metrics = model_best.val(
    data   = YAML_PATH,
    split  = 'test',
    device = DEVICE,
)

print('\nhasil evaluasi YOLOv8n di test set:')
print(f'  mAP@0.5        : {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95   : {metrics.box.map:.4f}')
print(f'  Precision      : {metrics.box.p[0]:.4f}')
print(f'  Recall         : {metrics.box.r[0]:.4f}')

Ultralytics 8.4.56  Python-3.12.3 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 433.6106.8 MB/s, size: 52.7 KB)
val: Scanning D:\malaria-detection\datasets\processed\labels\test.cache... 2803 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2803/2803  0.0s
val: D:\malaria-detection\datasets\processed\images\test\mpidb_0235.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 176/176 9.0it/s 19.6s0.1s
                   all       2803       3057      0.948      0.944      0.961      0.936
           parasitized       1406       1660      0.951      0.893      0.934      0.884
            uninfected       1397       1397      0.944      0.994      0.989      0.989
Speed: 1.1ms preprocess, 2.6ms inference, 0.0ms loss, 0.8ms postprocess per image
Resu

In [13]:
# ukur fps rata-rata di test set
import time
import cv2
from pathlib import Path

test_imgs = list(Path(summary['PATHS']['img_test']).glob('*.jpg'))[:100]

times = []
for img_path in test_imgs:
    img = cv2.imread(str(img_path))
    t0  = time.time()
    model_best.predict(img, verbose=False, device=DEVICE)
    times.append(time.time() - t0)

avg_ms  = (sum(times) / len(times)) * 1000
avg_fps = 1 / (sum(times) / len(times))

print(f'rata-rata inferensi : {avg_ms:.2f} ms')
print(f'rata-rata FPS       : {avg_fps:.1f}')

rata-rata inferensi : 11.34 ms
rata-rata FPS       : 88.2


## Simpan Hasil

In [14]:
total_params = sum(p.numel() for p in model_best.model.parameters())

# hitung f1 score dari precision dan recall
precision = float(metrics.box.p[0])
recall    = float(metrics.box.r[0])
f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

nb02_results = {
    'model'        : 'YOLOv8n',
    'best_weights' : best_weights,
    'map50'        : round(metrics.box.map50, 4),
    'map50_95'     : round(metrics.box.map, 4),
    'precision'    : round(precision, 4),
    'recall'       : round(recall, 4),
    'f1'           : round(f1, 4),
    'fps'          : round(avg_fps, 2),
    'avg_ms'       : round(avg_ms, 2),
    'total_params' : total_params,
}

result_path = os.path.join(PROJECT_ROOT, 'notebook02_results.json')
with open(result_path, 'w') as f:
    json.dump(nb02_results, f, indent=2)

print('hasil disimpan di:', result_path)
print(json.dumps(nb02_results, indent=2))

hasil disimpan di: d:\malaria-detection\notebook02_results.json
{
  "model": "YOLOv8n",
  "best_weights": "d:\\malaria-detection\\runs\\yolov8n\\train\\weights\\best.pt",
  "map50": 0.9613,
  "map50_95": 0.9365,
  "precision": 0.9515,
  "recall": 0.8934,
  "f1": 0.9215,
  "fps": 88.2,
  "avg_ms": 11.34,
  "total_params": 3006038
}


In [15]:
print('notebook 02 selesai')
print('best weights:', best_weights)

notebook 02 selesai
best weights: d:\malaria-detection\runs\yolov8n\train\weights\best.pt
